In [17]:
from pathlib import Path

notebook_path = Path().resolve()

In [18]:
cat_dir = notebook_path / 'raw/Cat'
dog_dir = notebook_path / 'raw/Dog'

In [19]:
cat_files = list(cat_dir.glob("*"))
dog_files = list(dog_dir.glob("*"))


In [20]:
print(f'Total cat images found: {len(cat_files)}')
print(f'Total dog images found: {len(dog_files)}')

Total cat images found: 12501
Total dog images found: 12501


In [21]:
# remove zero-byte and non jpeg files

from PIL import Image
corrupt_cats = []
corrupt_dogs = []

print("Checking out cat images for curruption ... ")
for p in cat_files:
    if p.stat().st_size == 0:
        corrupt_cats.append(p)
        continue
        
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        corrupt_cats.append(p)
        
        
print("Checking out dog images for curruption ... ")

for p in dog_files:
    if p.stat().st_size == 0:
        corrupt_dogs.append(p)
        continue
        
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        corrupt_dogs.append(p)


print(f"found {len(corrupt_cats)} corrupt cat images")
print(f"found {len(corrupt_dogs)} corrupt dog images")

Checking out cat images for curruption ... 
Checking out dog images for curruption ... 


/Users/psr/Documents/python/.venv/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


found 2 corrupt cat images
found 2 corrupt dog images


In [22]:
# clean data of cats and dogs and store it

import random

clean_cats = [p for p in cat_files if p not in corrupt_cats]
clean_dogs = [p for p in dog_files if p not in corrupt_dogs]

# shuffling data for randomness
random.seed(42)
random.shuffle(clean_cats)
random.shuffle(clean_dogs)

In [23]:
# do train test split (80 - 20)
from sklearn.model_selection import train_test_split

all_paths = clean_cats + clean_dogs
all_labels = [0] * len(clean_cats) +  [1] * len(clean_dogs)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2,
    random_state=43,
    stratify=all_labels  # This ensures an equal 50/50 mix of cats/dogs in both splits
)

print(f'Train set size: {len(train_paths)} images')
print(f'Validation set size: {len(val_paths)} images')

Train set size: 19998 images
Validation set size: 5000 images


In [24]:
# preprocessing pipeline for mobileNet
import torchvision.transforms as transforms

data_transformers = transforms.Compose([
    transforms.Resize((224,224)), # mobileNet format 224x224
    transforms.ToTensor(), # conversion of PIL image to pytorch tensor
    transforms.Normalize( # Standard imagesNet normalization for mean and std. deviation
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.24, 0.225]
    )
])

print("Transforms defined successfully!")

Transforms defined successfully!


In [25]:
from torch.utils.data import Dataset
from PIL import Image

class CatDogDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform =transform

    def __len__(self): # tells pytorch how many total images are in this dataset
        return len(self.image_paths)

    def __getitem__(self, idx):
        # pytorch asks for a item at specific index
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # open image
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

#instantiating training and validation datasets
train_dataset = CatDogDataset(train_paths, train_labels, transform=data_transformers)
val_dataset = CatDogDataset(val_paths, val_labels, transform=data_transformers)

In [26]:
# DataLoader of batch size = 32

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Train Loader ready with {len(train_loader)} batches.")
print(f"Validation Loader ready with {len(val_loader)} batches.")

Train Loader ready with 625 batches.
Validation Loader ready with 157 batches.


In [27]:
# Loading the mobileNet model
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# loading pre-trained moibleNet v3 with weights
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights = weights)

for param in model.parameters():
    param.requires_grad = False

# since mobileNet v3 has a input size of 1024 we only want to use 2 classes therefore we do-
in_features = model.classifier[3].in_features # how many signals are coming into mobileNet from previous layer ? « 1024
model.classifier[3] = nn.Linear(in_features, 2) # « output to only 2 nodes

print(model.classifier)

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=2, bias=True)
)


In [30]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss() # measures how wrong the models predictions are.

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr = 0.001)

print("Loss function and optimizer initialized")

Loss function and optimizer initialized


In [32]:
# testing a batch of 50 epoch
import time

model.train()

print("Starting test training pass ... ")
start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader):

    optimizer.zero_grad() #clear old gradients

    outputs = model(images) # feeding images into mobileNet

    loss = criterion(outputs, labels) #calculate losses

    loss.backward() # math to fix the gradients

    optimizer.step() # slightly fix the errors

    if (batch_idx + 1) % 10 == 0: # print status every 10 batches
        print(f"Batch [{batch_idx + 1}/50] | Loss: {loss.item():.4f}")

    if batch_idx + 1 == 200: # stop after 50 batches
        break

endtime = time.time()
print(f"Test pass finished in {endtime - start_time:.2f} seconds!")

Starting test training pass ... 
Batch [10/50] | Loss: 0.2068
Batch [20/50] | Loss: 0.1208
Batch [30/50] | Loss: 0.1624
Batch [40/50] | Loss: 0.2166
Batch [50/50] | Loss: 0.1121
Batch [60/50] | Loss: 0.2000
Batch [70/50] | Loss: 0.1863
Batch [80/50] | Loss: 0.2047
Batch [90/50] | Loss: 0.2501
Batch [100/50] | Loss: 0.3270
Batch [110/50] | Loss: 0.1851
Batch [120/50] | Loss: 0.1113
Batch [130/50] | Loss: 0.1213
Batch [140/50] | Loss: 0.1939
Batch [150/50] | Loss: 0.0868
Batch [160/50] | Loss: 0.1088
Batch [170/50] | Loss: 0.2088
Batch [180/50] | Loss: 0.2889
Batch [190/50] | Loss: 0.1803
Batch [200/50] | Loss: 0.1748
Test pass finished in 98.88 seconds!
